In [ ]:

from data_prep import get_smiles_df

df = get_smiles_df()

/Users/travisholt/github/trvslhlt/ift_6390_machine_learning/assignments/assignment_2/src/data_prep.py:59: DtypeWarning: Columns (0,6,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(URL)


In [ ]:
import torch 

vocab = sorted(set("".join(df["smiles"])))
char_to_idx = {c: i + 1 for i, c in enumerate(vocab)}

max_len = max(len(s) for s in df["smiles"]) # 163
X = torch.zeros(len(df), max_len, dtype=torch.long)
for i, s in enumerate(df["smiles"]):
    indices = [char_to_idx[c] for c in s]
    X[i, :len(indices)] = torch.tensor(indices)


In [17]:
from torch.nn import Embedding

EMBEDDING_DIM = 32
embedding = Embedding(num_embeddings=len(vocab) + 1, embedding_dim=EMBEDDING_DIM)
embedded_X = embedding(X)
embedded_X.shape

torch.Size([13131, 163, 32])

In [20]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class SmilesDataset(Dataset):
    def __init__(self, smiles_list, targets, char_to_idx):
        self.sequences = [torch.tensor([char_to_idx[c] for c in s], dtype=torch.long) for s in smiles_list]
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]
    
def collate_fn(batch):
    sequences, targets = zip(*batch)
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
    targets = torch.stack(targets)
    return padded_sequences, targets

In [21]:
dataset = SmilesDataset(df['smiles'].tolist(), df['target'].tolist(), char_to_idx)
loader = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)